## Retrieval QA
RAG Document Q&A System<br>
Retrieval Augmented Generation with LangChain and Groq<br>

This notebook demonstrates a complete RAG (Retrieval Augmented Generation) <br>
pipeline for answering questions about any PDF document.<br>

What this notebook does:<br>
1. **Loads** a PDF document from a URL
2. **Splits** it into manageable chunks
3. **Embeds** each chunk using HuggingFace's sentence transformers (runs locally)
4. **Stores** embeddings in ChromaDB vector database
5. **Retrieves** relevant chunks based on user questions
6. **Answers** questions using Groq's Llama model

Key concept — RAG:<br>
Instead of asking an LLM questions from its training data alone, <br>
RAG gives the LLM access to specific documents at query time.<br>
This means the LLM answers based on OUR documents, not general knowledge.<br>

Libraries used:<br>
- LangChain — orchestration framework<br>
- ChromaDB — vector database for storing embeddings<br>
- HuggingFace Sentence Transformers — local embedding model<br>
- Groq (Llama 3.3) — LLM for generating answers<br>

##### Imports and setup

In [11]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    api_key=os.getenv("GROQ_API_KEY")
)

##### Load and split PDF

In [12]:
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

text_splitter = CharacterTextSplitter(
    chunk_size=200, 
    chunk_overlap=20, 
    separator="\n"
)
chunks = text_splitter.split_documents(document)
print(f"Total chunks: {len(chunks)}")

Total chunks: 147


##### Embeddings

In [24]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"  # small, fast, free, runs locally
)
#The vector uses this 'embeddings' to convert text to numbers.
#Even this 'how it should work' can be configured by adding 'params' according to what we need.
#And then, whatever params we are using, we can even define 'TRUNCATE_INPUT_TOKENS' and 'RETURN_OPTIONS' according to us.
#For more elaboration, see file: '07.1_Embedding Models, Vector Stores, and Retrievers'.

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

##### Vector store and QA chain

In [23]:
docsearch = Chroma.from_documents(chunks, embeddings) #docsearch is the vector store here, which stores the 'chunks' and in the form of what it got from 'embeddings'

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", #Takes all retrieved chunks and stuffs them all into one promp
    retriever=docsearch.as_retriever(),
    return_source_documents=True  # let's see sources this time..
)


##### Asking questions

In [19]:
questions = [
    "What is this paper discussing?",
    "What is the purpose of MindGuide?",
    "What LangChain components does MindGuide use?",
    "What mental health problems does MindGuide solve?"
]

for question in questions:
    result = qa.invoke(question)
    print(f"\nQ: {question}")
    print(f"A: {result['result']}")
    print(f"Source: {result['source_documents'][0].page_content[:100]}...")
    print("---")


Q: What is this paper discussing?
A: This text appears to be discussing mental health, specifically the complexities of addressing mental health challenges and the interactions between individuals and a framework called LangChain, which seems to be an AI-powered system designed to facilitate conversations and provide support for mental health issues.
Source: throughout extraordinary demographic businesses and areas. 
However, what makes this situation even ...
---

Q: What is the purpose of MindGuide?
A: The purpose of MindGuide appears to be providing guidance and support to individuals in need, particularly those dealing with issues such as depression and anxiety.
Source: individuals in need of guidance and support in these critical 
areas. MindGuide relies on the capabi...
---

Q: What LangChain components does MindGuide use?
A: According to the provided context, MindGuide uses the following LangChain component: 

1. ChatModel (specifically Chat OpenAI) 

It may use other component

## Notes:
We have other options too other than 'stuff' such as:<br>
map_reduce — for large documents (Good when you have too many chunks to fit in one prompt):<br>
-Step 1: Send each chunk to LLM separately → get mini answer for each<br>
-Step 2: Combine all mini answers → send to LLM again → final answer<br>
refine — iterative approach (Good for nuanced answers that build on each other):<br>
-Step 1: Answer using chunk 1<br>
-Step 2: Refine that answer using chunk 2<br>
-Step 3: Refine again using chunk 3<br>
...until all chunks processed<br>
map_rerank — most precise (Good when you want the single most relevant answer):<br>
-Step 1: Send each chunk to LLM separately → get answer + confidence score<br>
-Step 2: Return the answer with highest confidence score<br>
When to use which:<br>
| Method | To use when: |
| :--- | :--- |
| stuff | Small documents, few chunks – simplest and fastest |
| map_reduce | Large documents, many chunks |
| refine | Need thorough, nuanced answers |
| map_rerank | Need the single most confident answer |